# No-Show Appointments — Improved Classification
**Improvements over baseline:**
- Rich datetime feature engineering
- Patient history aggregation features
- XGBoost + LightGBM + Voting Ensemble
- SMOTE for class imbalance
- Cross-validated evaluation (not just one split)
- Full metrics: Accuracy, Precision, Recall, F1, AUC-ROC

In [1]:
!pip install xgboost lightgbm

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   - -------------------------------------- 3.9/101.7 MB 19.5 MB/s eta 0:00:06
   -- ------------------------------------- 5.5/101.7 MB 12.9 MB/s eta 0:00:08
   --- ------------------------------------ 8.4/101.7 MB 13.0 MB/s eta 0:00:08
   --- ------------------------------------ 10.0/101.7 MB 11.5 MB/s eta 0:00:08
   ----- ---------------------------------- 12.8/101.7 MB 12.0 MB/s eta 0:00:08
   ----- ---------------------------------- 14.2/101.7 MB 11.0 MB/s eta 0:00:08
   ------ --------------------------------- 16.8/101.7 MB 11.4 MB/s eta 0:00:08
   ------- -------------------------------- 19.9/101.7 MB 11.5 MB/s eta 0:00:08
   -------- ------------------------------- 22.8/101.7 MB 11.6 MB/s eta 0:00:07
   ---------- ----------------------------- 25.4/101.7 MB 11.8 MB/s eta 0:00:07
   ----------- ---------------------------- 28.3/101.7


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ── 1. IMPORTS ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             confusion_matrix)
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from collections import Counter

# Install if needed: pip install xgboost lightgbm imbalanced-learn
import xgboost as xgb
import lightgbm as lgb

print('All libraries loaded successfully')

All libraries loaded successfully


In [4]:
# ── 2. LOAD DATA ─────────────────────────────────────────────────────────────
# Dataset: https://www.kaggle.com/datasets/joniarroba/noshowappointments
# Place KaggleV2-May-2016.csv in the same folder, or adjust path
df = pd.read_csv('Data/KaggleV2-May-2016.csv')

print(f'Shape: {df.shape}')
print(df.head(3).T)

Shape: (110527, 14)
                                   0                     1  \
PatientId           29872499824296.0     558997776694438.0   
AppointmentID                5642903               5642503   
Gender                             F                     M   
ScheduledDay    2016-04-29T18:38:08Z  2016-04-29T16:08:27Z   
AppointmentDay  2016-04-29T00:00:00Z  2016-04-29T00:00:00Z   
Age                               62                    56   
Neighbourhood        JARDIM DA PENHA       JARDIM DA PENHA   
Scholarship                        0                     0   
Hipertension                       1                     0   
Diabetes                           0                     0   
Alcoholism                         0                     0   
Handcap                            0                     0   
SMS_received                       0                     0   
No-show                           No                    No   

                                   2  
PatientId 

In [5]:
# ── 3. BASIC CLEANING ────────────────────────────────────────────────────────
# Fix column name inconsistency
df.rename(columns={'No-show': 'No_show', 'Hipertension': 'Hypertension'}, inplace=True)

# Parse datetimes
df['ScheduledDay'] = pd.to_datetime(df['ScheduledDay'])
df['AppointmentDay'] = pd.to_datetime(df['AppointmentDay'])

# Target: 1 = no-show, 0 = showed up
df['target'] = (df['No_show'] == 'Yes').astype(int)

# Drop rows with invalid age
df = df[df['Age'] >= 0].reset_index(drop=True)

print('Target distribution:')
print(df['target'].value_counts())
print(f'No-show rate: {df["target"].mean():.2%}')

Target distribution:
target
0    88207
1    22319
Name: count, dtype: int64
No-show rate: 20.19%


In [6]:
# ── 4. FEATURE ENGINEERING ───────────────────────────────────────────────────

# --- 4a. Datetime features ---
df['lead_days'] = (df['AppointmentDay'] - df['ScheduledDay']).dt.days
df['lead_days'] = df['lead_days'].clip(lower=0)  # fix negative values (data errors)

df['sched_dayofweek']  = df['ScheduledDay'].dt.dayofweek      # 0=Mon
df['appt_dayofweek']   = df['AppointmentDay'].dt.dayofweek
df['sched_hour']       = df['ScheduledDay'].dt.hour
df['appt_month']       = df['AppointmentDay'].dt.month
df['appt_week']        = df['AppointmentDay'].dt.isocalendar().week.astype(int)
df['is_weekend_appt']  = (df['appt_dayofweek'] >= 5).astype(int)
df['same_day']         = (df['lead_days'] == 0).astype(int)

# Lead day bins
df['lead_bin'] = pd.cut(df['lead_days'],
                        bins=[-1, 0, 3, 7, 14, 30, 60, 1000],
                        labels=[0, 1, 2, 3, 4, 5, 6]).astype(int)

# --- 4b. Age features ---
df['age_bin'] = pd.cut(df['Age'],
                       bins=[-1, 2, 12, 17, 35, 55, 75, 200],
                       labels=[0, 1, 2, 3, 4, 5, 6]).astype(int)
df['is_child']  = (df['Age'] <= 12).astype(int)
df['is_senior'] = (df['Age'] >= 65).astype(int)

# --- 4c. Comorbidity score ---
df['comorbidity_score'] = (df['Hypertension'] + df['Diabetes'] +
                           df['Alcoholism'] + df['Handcap'])

# --- 4d. Patient history features (key improvement!) ---
# Sort by patient and scheduled date to compute history correctly
df.sort_values(['PatientId', 'ScheduledDay'], inplace=True)
df.reset_index(drop=True, inplace=True)

# Number of previous appointments per patient (cumulative count - 1)
df['prev_appt_count'] = df.groupby('PatientId').cumcount()

# Previous no-show rate per patient (rolling, excludes current row)
df['prev_noshow_count'] = df.groupby('PatientId')['target'].cumsum() - df['target']
df['prev_noshow_rate'] = np.where(
    df['prev_appt_count'] > 0,
    df['prev_noshow_count'] / df['prev_appt_count'],
    0.0
)

# --- 4e. Neighbourhood no-show rate (target encoding) ---
# Use global mean to avoid leakage
neighbourhood_rate = df.groupby('Neighbourhood')['target'].mean()
df['neighbourhood_noshow_rate'] = df['Neighbourhood'].map(neighbourhood_rate)

# --- 4f. SMS reminder interaction ---
df['sms_long_lead'] = df['SMS_received'] * (df['lead_days'] > 7).astype(int)

print('Feature engineering done.')
print(f'Total features available: {df.shape[1]}')

Feature engineering done.
Total features available: 33


In [7]:
# ── 5. PREPARE FEATURE MATRIX ────────────────────────────────────────────────
features = [
    # Core
    'Age', 'age_bin', 'is_child', 'is_senior',
    # Datetime
    'lead_days', 'lead_bin', 'same_day',
    'sched_dayofweek', 'appt_dayofweek', 'sched_hour',
    'appt_month', 'appt_week', 'is_weekend_appt',
    # Medical
    'Hypertension', 'Diabetes', 'Alcoholism', 'Handcap',
    'comorbidity_score',
    # Reminder
    'SMS_received', 'sms_long_lead',
    # History
    'prev_appt_count', 'prev_noshow_count', 'prev_noshow_rate',
    # Location
    'neighbourhood_noshow_rate',
    # Scholarship
    'Scholarship',
]

X = df[features].astype('float64')
y = df['target']

print(f'Feature matrix shape: {X.shape}')
print(f'Class balance: {Counter(y)}')

Feature matrix shape: (110526, 25)
Class balance: Counter({0: 88207, 1: 22319})


In [8]:
# ── 6. TRAIN/TEST SPLIT + SMOTE ──────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Before SMOTE: {Counter(y_train)}')
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
print(f'After SMOTE:  {Counter(y_train_res)}')

Before SMOTE: Counter({0: 70565, 1: 17855})
After SMOTE:  Counter({1: 70565, 0: 70565})


In [9]:
# ── 7. DEFINE MODELS ─────────────────────────────────────────────────────────

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

lgb_model = lgb.LGBMClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_split=10,
    random_state=42,
    n_jobs=-1
)

# Soft-voting ensemble
ensemble = VotingClassifier(
    estimators=[('xgb', xgb_model), ('lgb', lgb_model), ('rf', rf_model)],
    voting='soft'
)

print('Models defined.')

Models defined.


In [10]:
# ── 8. TRAIN AND EVALUATE ALL MODELS ─────────────────────────────────────────
models = {
    'XGBoost':  xgb_model,
    'LightGBM': lgb_model,
    'Random Forest': rf_model,
    'Voting Ensemble': ensemble
}

results = {}

for name, model in models.items():
    print(f'\nTraining {name}...')
    model.fit(X_train_res, y_train_res)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    results[name] = {
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall':    recall_score(y_test, y_pred),
        'F1':        f1_score(y_test, y_pred),
        'AUC-ROC':   roc_auc_score(y_test, y_proba)
    }
    print(f"  Accuracy: {results[name]['Accuracy']:.4f}  "
          f"F1: {results[name]['F1']:.4f}  "
          f"AUC: {results[name]['AUC-ROC']:.4f}")

print('\n── Summary ──────────────────────────────────────────')
results_df = pd.DataFrame(results).T.round(4)
print(results_df)


Training XGBoost...
  Accuracy: 0.8026  F1: 0.2174  AUC: 0.7482

Training LightGBM...
  Accuracy: 0.8021  F1: 0.2021  AUC: 0.7470

Training Random Forest...
  Accuracy: 0.7472  F1: 0.4033  AUC: 0.7376

Training Voting Ensemble...
  Accuracy: 0.7967  F1: 0.2927  AUC: 0.7464

── Summary ──────────────────────────────────────────
                 Accuracy  Precision  Recall      F1  AUC-ROC
XGBoost            0.8026     0.5450  0.1358  0.2174   0.7482
LightGBM           0.8021     0.5437  0.1241  0.2021   0.7470
Random Forest      0.7472     0.3853  0.4232  0.4033   0.7376
Voting Ensemble    0.7967     0.4921  0.2083  0.2927   0.7464


In [11]:
# ── 9. CROSS-VALIDATION ON BEST MODEL ────────────────────────────────────────
print('5-fold CV on XGBoost (on full resampled data)...')

# Resample full dataset for CV
X_res, y_res = sm.fit_resample(X, y)

cv_scores = cross_val_score(
    xgb.XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        use_label_encoder=False, eval_metric='logloss', random_state=42
    ),
    X_res, y_res,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1',
    n_jobs=-1
)

print(f'CV F1 scores: {cv_scores.round(4)}')
print(f'Mean F1: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

5-fold CV on XGBoost (on full resampled data)...
CV F1 scores: [0.8508 0.8541 0.8534 0.8551 0.8481]
Mean F1: 0.8523 ± 0.0025


In [12]:
# ── 10. FEATURE IMPORTANCE (XGBoost) ─────────────────────────────────────────
importance_df = pd.DataFrame({
    'Feature':    features,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

print('Top 15 most important features:')
print(importance_df.head(15).to_string(index=False))

Top 15 most important features:
          Feature  Importance
         same_day    0.258962
     SMS_received    0.097569
      Scholarship    0.080229
     Hypertension    0.071441
         lead_bin    0.063057
  sched_dayofweek    0.054553
comorbidity_score    0.049555
prev_noshow_count    0.045340
  prev_appt_count    0.044261
        appt_week    0.034996
   appt_dayofweek    0.033643
       appt_month    0.030968
       sched_hour    0.020767
 prev_noshow_rate    0.020415
         Diabetes    0.015952


In [13]:
# ── 11. DETAILED CLASSIFICATION REPORT (Best model) ──────────────────────────
best_pred = ensemble.predict(X_test)
print('Classification Report — Voting Ensemble:')
print(classification_report(y_test, best_pred, target_names=['Showed Up', 'No-Show']))
print('Confusion Matrix:')
print(confusion_matrix(y_test, best_pred))

Classification Report — Voting Ensemble:
              precision    recall  f1-score   support

   Showed Up       0.83      0.95      0.88     17642
     No-Show       0.49      0.21      0.29      4464

    accuracy                           0.80     22106
   macro avg       0.66      0.58      0.59     22106
weighted avg       0.76      0.80      0.76     22106

Confusion Matrix:
[[16682   960]
 [ 3534   930]]


In [17]:
# ── 12. PREDICT FOR A NEW PATIENT ────────────────────────────────────────────

def predict_noshow(patient_data: dict, model, feature_list):
    """
    patient_data: dict with raw input values
    model: trained model (e.g. ensemble)
    feature_list: the 'features' list used during training
    """
    # --- Parse dates ---
    scheduled  = pd.to_datetime(patient_data['ScheduledDay'])
    appointmt  = pd.to_datetime(patient_data['AppointmentDay'])

    lead_days  = max((appointmt - scheduled).days, 0)
    age        = patient_data['Age']

    # --- Build the same features used in training ---
    row = {
        # Core
        'Age':                      age,
        'age_bin':                  pd.cut([age], bins=[-1,2,12,17,35,55,75,200], labels=[0,1,2,3,4,5,6]).astype(int)[0],
        'is_child':                 int(age <= 12),
        'is_senior':                int(age >= 65),

        # Datetime
        'lead_days':                lead_days,
        'lead_bin':                 pd.cut([lead_days], bins=[-1,0,3,7,14,30,60,1000], labels=[0,1,2,3,4,5,6]).astype(int)[0],
        'same_day':                 int(lead_days == 0),
        'sched_dayofweek':          scheduled.dayofweek,
        'appt_dayofweek':           appointmt.dayofweek,
        'sched_hour':               scheduled.hour,
        'appt_month':               appointmt.month,
        'appt_week':                appointmt.isocalendar()[1],
        'is_weekend_appt':          int(appointmt.dayofweek >= 5),

        # Medical
        'Hypertension':             patient_data['Hypertension'],
        'Diabetes':                 patient_data['Diabetes'],
        'Alcoholism':               patient_data['Alcoholism'],
        'Handcap':                  patient_data['Handcap'],
        'comorbidity_score':        patient_data['Hypertension'] + patient_data['Diabetes'] +
                                    patient_data['Alcoholism'] + patient_data['Handcap'],

        # Reminder
        'SMS_received':             patient_data['SMS_received'],
        'sms_long_lead':            patient_data['SMS_received'] * int(lead_days > 7),

        # History — use 0 if it's a brand new patient
        'prev_appt_count':          patient_data.get('prev_appt_count', 0),
        'prev_noshow_count':        patient_data.get('prev_noshow_count', 0),
        'prev_noshow_rate':         patient_data.get('prev_noshow_rate', 0.0),

        # Location — use global mean if neighbourhood is unknown
        'neighbourhood_noshow_rate': patient_data.get('neighbourhood_noshow_rate',
                                                       df['target'].mean()),
        # Scholarship
        'Scholarship':              patient_data['Scholarship'],
    }

    input_df = pd.DataFrame([row])[feature_list].astype('float64')

    prob      = model.predict_proba(input_df)[0][1]
    prediction = 'NO-SHOW ⚠️' if prob >= 0.5 else 'WILL SHOW UP ✅'

    print(f"Prediction  : {prediction}")
    print(f"No-show prob: {prob:.2%}")
    return prob


# ── Example usage ─────────────────────────────────────────────────────────────
new_patient = {
    'ScheduledDay':   '2016-05-10 08:30:00',
    'AppointmentDay': '2016-05-25 00:00:00',
    'Age':            45,
    'Hypertension':   1,
    'Diabetes':       0,
    'Alcoholism':     0,
    'Handcap':        0,
    'SMS_received':   1,
    'Scholarship':    0,

    # If the patient has visited before, fill these in:
    'prev_appt_count':   3,
    'prev_noshow_count': 0,
    'prev_noshow_rate':  0,   # skipped 2 out of 3 previous visits

    # Average no-show rate for their neighbourhood (from training data)
    'neighbourhood_noshow_rate': 0.50,
}

predict_noshow(new_patient, ensemble, features)

Prediction  : WILL SHOW UP ✅
No-show prob: 30.73%


0.3073070984111352